# Determining the Fair Spread of a CDS Tranche

An analysis of pricing a CDS Index using its intrinsic value

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from financepy.utils import *
from financepy.products.rates.ibor_deposit import IborDeposit
from financepy.products.rates.ibor_swap import IborSwap
from financepy.market.curves.ibor_single_curve import IborSingleCurve
from financepy.market.curves.cds_curve import CDSCurve
from financepy.products.credit.cds import CDS
from financepy.products.credit.cds_tranche import CDSTranche, DefaultLossDbnAlgoTypes
from financepy.products.credit.cds_index_portfolio import CDSIndexPortfolio



#############################################################
#  FINANCEPY Version 1.1.2 - Built on 25 Sep 2026 at 14:07  #
#  This software is distributed FREE AND WITHOUT WARRANTY   #
#  Report issues at https://github.com/domokane/FinancePy   #
#############################################################



In [3]:
value_dt = Date(2, 8, 2007)
settle_dt = value_dt.add_weekdays(1)

## Build Ibor Curve

In [4]:
dc_type = DayCountTypes.THIRTY_E_360_ISDA
depos = []
depo = IborDeposit(value_dt, "1D", 0.0500, dc_type); depos.append(depo)
depo = IborDeposit(settle_dt, "1D", 0.0500, dc_type); depos.append(depo)

fixed_freq = FrequencyTypes.SEMI_ANNUAL
swap_type = SwapTypes.PAY
swap1 = IborSwap(settle_dt,"1Y",swap_type,0.0502,fixed_freq,dc_type)
swap2 = IborSwap(settle_dt,"2Y",swap_type,0.0502,fixed_freq,dc_type)
swap3 = IborSwap(settle_dt,"3Y",swap_type,0.0501,fixed_freq,dc_type)
swap4 = IborSwap(settle_dt,"4Y",swap_type,0.0502,fixed_freq,dc_type)
swap5 = IborSwap(settle_dt,"5Y",swap_type,0.0501,fixed_freq,dc_type)
swaps = [swap1,swap2,swap3,swap4,swap5]

libor_curve = IborSingleCurve(value_dt, depos, [], swaps)

We treat an index as a CDS contract with a flat CDS curve at the CDS index spread for the same maturity

## Create the Underlying CDS Index Portfolio

In [5]:
step_in_dt = value_dt.add_weekdays(1)

In [6]:
maturity_3yr = value_dt.next_cds_date(36)
maturity_5yr = value_dt.next_cds_date(60)
maturity_7yr = value_dt.next_cds_date(84)
maturity_10yr = value_dt.next_cds_date(120)

### Heterogeneous Curves

In [7]:
# Move into the local folder of notebook
%pwd
%cd examples/notebooks/products/credit/
%pwd


[WinError 3] The system cannot find the path specified: 'examples/notebooks/products/credit/'
C:\Users\Dominic\Dropbox\Desktop\RESEARCH_DB\FinancePy\Code\financepy-git\examples\notebooks\products\credit


'C:\\Users\\Dominic\\Dropbox\\Desktop\\RESEARCH_DB\\FinancePy\\Code\\financepy-git\\examples\\notebooks\\products\\credit'

In [8]:
f = open('.//data//CDX_NA_IG_S7_SPREADS.csv', 'r')
data = f.readlines()
heteroIssuerCurves = []

num_credits = len(data) - 1  # The file has a header

for row in data[1:]:
    splitRow = row.split(",")
    spd3Y = float(splitRow[1]) / 10000.0
    spd5Y = float(splitRow[2]) / 10000.0
    spd7Y = float(splitRow[3]) / 10000.0
    spd10Y = float(splitRow[4]) / 10000.0
    recovery_rate = float(splitRow[5])
    cds3Y = CDS(step_in_dt, maturity_3yr, spd3Y)
    cds5Y = CDS(step_in_dt, maturity_5yr, spd5Y)
    cds7Y = CDS(step_in_dt, maturity_7yr, spd7Y)
    cds10Y = CDS(step_in_dt, maturity_10yr, spd10Y)
    cds_contracts = [cds3Y, cds5Y, cds7Y, cds10Y]
    issuer_curve = CDSCurve(value_dt, cds_contracts, libor_curve, recovery_rate)
    heteroIssuerCurves.append(issuer_curve)

### Homogeneous Curves 

Calculate the average spread of the heterogeneous portfolio

In [9]:
homoIssuerCurves = []
num_credits = 125
recovery_rate = 0.40

In [10]:
cdsIndex = CDSIndexPortfolio()

In [11]:
spd3Y = cdsIndex.intrinsic_spread(value_dt, step_in_dt, maturity_3yr, heteroIssuerCurves)
spd5Y = cdsIndex.intrinsic_spread(value_dt, step_in_dt, maturity_5yr, heteroIssuerCurves)
spd7Y = cdsIndex.intrinsic_spread(value_dt, step_in_dt, maturity_7yr, heteroIssuerCurves)
spd10Y = cdsIndex.intrinsic_spread(value_dt, step_in_dt, maturity_10yr, heteroIssuerCurves)

In [12]:
print("Homogeneous curve 3Y:", spd3Y*10000)
print("Homogeneous curve 5Y:", spd5Y*10000)
print("Homogeneous curve 7Y:", spd7Y*10000)
print("Homogeneous curve 10Y:", spd10Y*10000)

Homogeneous curve 3Y: 19.678532093425247
Homogeneous curve 5Y: 35.53750784293198
Homogeneous curve 7Y: 49.00800186200032
Homogeneous curve 10Y: 61.40712247056099


In [13]:
for row in range(0,num_credits):
    cds3Y = CDS(step_in_dt, maturity_3yr, spd3Y)
    cds5Y = CDS(step_in_dt, maturity_5yr, spd5Y)
    cds7Y = CDS(step_in_dt, maturity_7yr, spd7Y)
    cds10Y = CDS(step_in_dt, maturity_10yr, spd10Y)
    cds_contracts = [cds3Y, cds5Y, cds7Y, cds10Y]
    issuer_curve = CDSCurve(value_dt, cds_contracts, libor_curve, recovery_rate)
    homoIssuerCurves.append(issuer_curve)

## Define the Tranches

In [14]:
trancheMaturity = maturity_5yr
tranche1 = CDSTranche(value_dt, trancheMaturity, 0.00, 0.03)
tranche2 = CDSTranche(value_dt, trancheMaturity, 0.03, 0.06)
tranche3 = CDSTranche(value_dt, trancheMaturity, 0.06, 0.09)
tranche4 = CDSTranche(value_dt, trancheMaturity, 0.09, 0.12)
tranche5 = CDSTranche(value_dt, trancheMaturity, 0.12, 0.22)
tranche6 = CDSTranche(value_dt, trancheMaturity, 0.22, 0.60)
tranche7 = CDSTranche(value_dt, trancheMaturity, 0.00, 0.60)
tranche8 = CDSTranche(value_dt, trancheMaturity, 0.00, 1.00)

In [15]:
tranches = [tranche1, tranche2, tranche3, tranche4, tranche5, tranche6, tranche7, tranche8]

In [16]:
corr1 = 0.30
corr2 = 0.30
upfront = 0.0
spd = 0.0

## Homogeneous Portfolio Results

In [17]:
print("%50s %5s %9s %9s %12s"% ("Method", "NumPts", "k_1", "k_2", "SPD(BPS)"))
for tranche in tranches:
    for method in DefaultLossDbnAlgoTypes:
        for num_points in [50]:
            v = tranche.value_bc(value_dt,homoIssuerCurves,upfront,spd,corr1,corr2,num_points,method)
            print("%50s %5d %9.5f %9.5f %12.6f"% (method, num_points, tranche.k1*100, tranche.k2*100, v[3] * 10000))
    print("=============================================================================================")

                                            Method NumPts       k_1       k_2     SPD(BPS)
                 DefaultLossDbnAlgoTypes.RECURSION    50   0.00000   3.00000   875.465517
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50   0.00000   3.00000   875.465517
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50   0.00000   3.00000   908.964756
                       DefaultLossDbnAlgoTypes.LHP    50   0.00000   3.00000   914.783647
                 DefaultLossDbnAlgoTypes.RECURSION    50   3.00000   6.00000   239.598793
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50   3.00000   6.00000   239.598793
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50   3.00000   6.00000   239.287847
                       DefaultLossDbnAlgoTypes.LHP    50   3.00000   6.00000   226.823997
                 DefaultLossDbnAlgoTypes.RECURSION    50   6.00000   9.00000   102.119820
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50   6.00000   9.00000   102.119820
         

                 DefaultLossDbnAlgoTypes.RECURSION    50   9.00000  12.00000    49.205418
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50   9.00000  12.00000    49.205418
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50   9.00000  12.00000    49.026806
                       DefaultLossDbnAlgoTypes.LHP    50   9.00000  12.00000    45.128032
                 DefaultLossDbnAlgoTypes.RECURSION    50  12.00000  22.00000    14.016682
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50  12.00000  22.00000    14.016682
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50  12.00000  22.00000    13.997424
                       DefaultLossDbnAlgoTypes.LHP    50  12.00000  22.00000    12.628064
                 DefaultLossDbnAlgoTypes.RECURSION    50  22.00000  60.00000     0.489274
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50  22.00000  60.00000     0.489274
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50  22.00000  60.00000     0.488314
          

                       DefaultLossDbnAlgoTypes.LHP    50   0.00000  60.00000    59.212809
                 DefaultLossDbnAlgoTypes.RECURSION    50   0.00000 100.00000    35.374656
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50   0.00000 100.00000    35.374656
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50   0.00000 100.00000    35.942907
                       DefaultLossDbnAlgoTypes.LHP    50   0.00000 100.00000    35.374648


## Heterogeneous Portfolio Results

In [18]:
print("%50s %5s %9s %9s %12s"% ("Method", "NumPts", "k_1", "k_2", "SPD(BPS)"))

for tranche in tranches:
    for method in DefaultLossDbnAlgoTypes:
        for num_points in [50]:
            v = tranche.value_bc(value_dt,heteroIssuerCurves,upfront,spd,corr1,corr2,num_points,method)
            print("%50s %5d  %9.5f %9.5f %12.6f"% (method, num_points, tranche.k1*100, tranche.k2*100, v[3] * 10000))
    print("=============================================================================================")

                                            Method NumPts       k_1       k_2     SPD(BPS)
                 DefaultLossDbnAlgoTypes.RECURSION    50    0.00000   3.00000   949.984575
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50    0.00000   3.00000   949.945289
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50    0.00000   3.00000   983.652284
                       DefaultLossDbnAlgoTypes.LHP    50    0.00000   3.00000   915.153159
                 DefaultLossDbnAlgoTypes.RECURSION    50    3.00000   6.00000   230.904156
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50    3.00000   6.00000   230.911737
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50    3.00000   6.00000   230.459833
                       DefaultLossDbnAlgoTypes.LHP    50    3.00000   6.00000   226.823232
                 DefaultLossDbnAlgoTypes.RECURSION    50    6.00000   9.00000    87.192645
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50    6.00000   9.00000    87.258258

                  DefaultLossDbnAlgoTypes.GAUSSIAN    50    6.00000   9.00000    86.831945
                       DefaultLossDbnAlgoTypes.LHP    50    6.00000   9.00000    94.900523


                 DefaultLossDbnAlgoTypes.RECURSION    50    9.00000  12.00000    37.474570
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50    9.00000  12.00000    37.377574
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50    9.00000  12.00000    37.310758
                       DefaultLossDbnAlgoTypes.LHP    50    9.00000  12.00000    45.118503
                 DefaultLossDbnAlgoTypes.RECURSION    50   12.00000  22.00000     8.937954
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50   12.00000  22.00000     8.954019
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50   12.00000  22.00000     8.924429
                       DefaultLossDbnAlgoTypes.LHP    50   12.00000  22.00000    12.624256


                 DefaultLossDbnAlgoTypes.RECURSION    50   22.00000  60.00000     0.212493
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50   22.00000  60.00000     0.212460
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50   22.00000  60.00000     0.212017
                       DefaultLossDbnAlgoTypes.LHP    50   22.00000  60.00000     0.417752


                 DefaultLossDbnAlgoTypes.RECURSION    50    0.00000  60.00000    59.212799
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50    0.00000  60.00000    59.212799
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50    0.00000  60.00000    60.116426
                       DefaultLossDbnAlgoTypes.LHP    50    0.00000  60.00000    59.212658
                 DefaultLossDbnAlgoTypes.RECURSION    50    0.00000 100.00000    35.374303
         DefaultLossDbnAlgoTypes.ADJUSTED_BINOMIAL    50    0.00000 100.00000    35.374303
                  DefaultLossDbnAlgoTypes.GAUSSIAN    50    0.00000 100.00000    35.907041
                       DefaultLossDbnAlgoTypes.LHP    50    0.00000 100.00000    35.374285


Copyright (c) 2020 Dominic O'Kane